<img src="https://wamugii.kento.rw/logo-emblem-128.png" width="56" align="right" alt="WAMUGII">

**Math & Python for AI** · Week 2 · Working with Data · Lesson 9 of 24

# Cleaning Messy Data

**How to use this notebook:** run the code cells **in order**, from top to bottom (press ▶ on each cell, or use **Runtime → Run all**). You can change any code and run it again. To keep your changes, use **File → Save a copy in Drive**.

The lesson, quiz and certificate are on [WAMUGII](https://wamugii.kento.rw/courses/math-python-for-ai).

**Learning objective:** Find and fix the most common data problems — missing values, duplicates, wrong types, messy text and impossible values — and document what you changed.

## Real data is always messy

Data typed by people, exported from systems or joined from several sources is never perfect. Data scientists often spend more time **cleaning** data than modelling it. A model trained on dirty data learns dirty patterns.

## A messy dataset

Run this cell. It builds a small file the way it might arrive from a shop's records:

In [ ]:
import numpy as np
import pandas as pd

raw = pd.DataFrame({
    "date": ["2026-03-01", "2026-03-01", "2026-03-02", "2026-03-02", "2026-03-03",
             "2026-03-03", "2026-03-04", "not recorded", "2026-03-05", "2026-03-05"],
    "district": ["Gasabo", "gasabo ", "Kicukiro", "KICUKIRO", "Nyarugenge",
                 "Gasabo", None, "Kicukiro", "Nyarugenge", "Nyarugenge"],
    "product": ["Rice 5kg", "Rice 5kg", "Milk 1L", "Sugar 1kg", "Milk 1L",
                "Airtime 1000", "Milk 1L", "Rice 5kg", "Sugar 1kg", "Sugar 1kg"],
    "quantity": ["4", "4", "6", "two", "3", "5", "2", "1", "-3", "2"],
    "unit_price": [7500, 7500, 800, 1600, 800, 1000, 800, 7500, 1600, None],
})
print(raw)

## Step 1 — Inspect

In [ ]:
raw.info()
print(raw.isna().sum())          # missing values per column
print(raw["district"].unique())  # spelling variants

## Step 2 — Remove exact duplicates

In [ ]:
print(raw.duplicated().sum())    # 0: the copies only match after cleaning
df = raw.copy()
df["district"] = df["district"].str.strip().str.title()   # fix text first
df = df.drop_duplicates()
print(len(raw), "->", len(df))

Clean the text **before** dropping duplicates: `"Gasabo"` and `"gasabo "` only look the same after cleaning.

## Step 3 — Fix types

In [ ]:
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce")
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df.info()

`errors="coerce"` turns values that cannot be converted (`"two"`, `"not recorded"`) into **missing** (`NaN` / `NaT`) instead of crashing. Now you can decide what to do with them.

## Step 4 — Impossible values

In [ ]:
print(df[df["quantity"] <= 0])
df.loc[df["quantity"] <= 0, "quantity"] = np.nan    # a sale cannot be negative

Ask: *is this value possible in the real world?* Negative quantities, ages of 340, or prices of 0 for paid items are errors.

## Step 5 — Handle missing values

There is no single right answer. Options:

| Option | Code | When it makes sense |
|---|---|---|
| Drop the rows | `df.dropna(subset=["quantity"])` | Few rows affected, and the value is essential |
| Fill with a fixed value | `df["district"].fillna("Unknown")` | Categories, where "unknown" is honest |
| Fill from known facts | look up the product's price | The correct value can be recovered |
| Fill with mean/median | `df["x"].fillna(df["x"].median())` | Numbers, when few are missing — use with care |

In [ ]:
prices = {"Rice 5kg": 7500, "Milk 1L": 800, "Sugar 1kg": 1600, "Airtime 1000": 1000}
df["unit_price"] = df["unit_price"].fillna(df["product"].map(prices))
df["district"] = df["district"].fillna("Unknown")
df = df.dropna(subset=["quantity", "date"])
df["quantity"] = df["quantity"].astype(int)
df["total"] = df["quantity"] * df["unit_price"]
print(df)

## Step 6 — Write a cleaning log

Professionals record every change, so others can check and repeat it:

```text
- Standardised district names (strip spaces, Title Case): 2 values changed
- Removed 1 exact duplicate row
- quantity "two" and date "not recorded" could not be read -> rows dropped (2)
- quantity -3 is impossible -> row dropped (1)
- 1 missing unit_price filled from the official price list
- 1 missing district set to "Unknown"
```

## Key terms

- **Missing value (NaN)** — an empty cell; `isna()` finds them.
- **Duplicate** — a row that appears more than once.
- **Type conversion** — e.g. `pd.to_numeric`, `pd.to_datetime`.
- **coerce** — turn unreadable values into missing instead of failing.
- **Imputation** — filling missing values with an estimate.
- **Cleaning log** — a written record of every cleaning step.

**Takeaway:** Inspect, standardise text, remove duplicates, fix types, remove impossible values, handle missing values thoughtfully — and write down every step.

## Try it yourself

1. Run all the steps and check that the final table has no missing values (`df.isna().sum()`).
2. Change one decision: instead of dropping the row with quantity `"two"`, set it to 2. How does the total income change?
3. Write your own cleaning log for your version, in plain sentences.

---
Finished? Go back to [Math & Python for AI on WAMUGII](https://wamugii.kento.rw/courses/math-python-for-ai), tick the lesson as done and take the quiz.

*© WAMUGII TECH SOLUTIONS · Original course material for WAMUGII AI Program students.*